In [1]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
sys.path.append('..')
from src.LSTM import LSTM
from src.metrics import evaluate_predictions

PROC_DIR = '../data/processed'
PRED_DIR = '../data/predictions'
REPORT_DIR = '../data/reports'
MODEL_DIR = '../data/models'
VOCAB_SIZE = 1001
ITEM_DIM = VOCAB_SIZE - 1
MAX_SEQ_LEN = 10
BATCH_SIZE = 256
MAX_EPOCHS = 40
PATIENCE = 5
RECENCY_DECAY = 0.7
SEED = 42
K_VALUES = [1, 3, 5, 9]
for d in [PRED_DIR, REPORT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [2]:
X_padded = np.load(f'{PROC_DIR}/X_padded.npy').astype(np.uint8)
Y = np.load(f'{PROC_DIR}/y_target.npy').astype(np.float32)
splits = np.load(f'{PROC_DIR}/splits.npz')
train_ex, val_ex, test_ex = splits['train'], splits['validation'], splits['test']
users = test_ex[:, 0]
H_raw = X_padded[:, :, 1:]
S_full = np.concatenate([X_padded, Y[:, None, :].astype(np.uint8)], axis=1)
print(f"History without RARE column: {H_raw.shape}, full sequences: {S_full.shape}")

History without RARE column: (14074, 10, 1000), full sequences: (14074, 11, 1001)


In [3]:
def raw_window(u, p):
    L = min(p, MAX_SEQ_LEN)
    x = np.zeros((MAX_SEQ_LEN, ITEM_DIM), dtype=np.uint8)
    x[MAX_SEQ_LEN - L:] = H_raw[u, p - L:p]
    return x


def history_freq_window(u, p, decay=RECENCY_DECAY):
    L = min(p, MAX_SEQ_LEN)
    hist_baskets = S_full[u, p - L:p, 1:].astype(np.float32)
    weights = (decay ** np.arange(L - 1, -1, -1, dtype=np.float32))[:, None]
    return np.log1p((weights * hist_baskets).sum(axis=0)).astype(np.float32)


def materialize(examples, drop_empty_targets=False):
    X = np.stack([raw_window(u, p) for u, p in examples])
    H = np.stack([history_freq_window(u, p) for u, p in examples]).astype(np.float32)
    y = S_full[examples[:, 0], examples[:, 1]].astype(np.float32)
    if drop_empty_targets:
        valid = y[:, 1:].sum(axis=1) > 0
        X, H, y = X[valid], H[valid], y[valid]
    return X, H, y


Xtr, Htr, ytr = materialize(train_ex, drop_empty_targets=True)
Xva, Hva, yva = materialize(val_ex, drop_empty_targets=False)
Xte, Hte, yte = materialize(test_ex, drop_empty_targets=False)
ytr_items = ytr[:, 1:]
yva_items = yva[:, 1:]
print(f"Train {Xtr.shape}, H: {Htr.shape} | validation {Xva.shape} | test {Xte.shape}")

Train (32487, 10, 1000), H: (32487, 1000) | validation (14074, 10, 1000) | test (14074, 10, 1000)


In [4]:
def make_ds(X, H, y, shuffle):
    def gen():
        idx = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            b = idx[i:i + BATCH_SIZE]
            yield (X[b].astype(np.float32), H[b]), y[b]
    spec = (
        (
            tf.TensorSpec((None, MAX_SEQ_LEN, ITEM_DIM), tf.float32),
            tf.TensorSpec((None, ITEM_DIM), tf.float32),
        ),
        tf.TensorSpec((None, ITEM_DIM), tf.float32),
    )
    return tf.data.Dataset.from_generator(gen, output_signature=spec).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(Xtr, Htr, ytr_items, shuffle=True)
val_ds = make_ds(Xva, Hva, yva_items, shuffle=False)


def to_full(scores):
    return np.concatenate([np.zeros((len(scores), 1), np.float32), scores], axis=1)

In [5]:
with open(f'{REPORT_DIR}/lstm_training_report.json') as f:
    cfg = json.load(f)['selected_config']
print(f"Using the RBM model's selected config: {cfg}")

Xva_f32 = Xva.astype(np.float32)


class ValHitRatio(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = -1.0
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        scores = to_full(self.model.predict([Xva_f32, Hva], batch_size=BATCH_SIZE, verbose=0))
        res = evaluate_predictions(scores, yva, K_VALUES)
        mhr = float(np.mean([res[k]['HitRatio'] for k in K_VALUES]))
        print(f"    epoch {epoch + 1:>2} | val mean HR={mhr:.4f}", flush=True)
        if mhr > self.best:
            self.best = mhr
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


net = LSTM(
    sequence_length=MAX_SEQ_LEN,
    input_dim=ITEM_DIM,
    output_dim=ITEM_DIM,
    hidden_units=cfg['hidden_units'],
    dropout=cfg['dropout'],
    learning_rate=cfg['learning_rate'],
    pos_weight=cfg.get('pos_weight', 15.0),
    use_history_residual=True,
)
callbacks = [
    ValHitRatio(),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5
    ),
]
hist = net.model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

Using the RBM model's selected config: {'learning_rate': 0.001, 'dropout': 0.3, 'hidden_units': 256, 'pos_weight': 20.0}

Epoch 1/40


c:\Users\krpri\anaconda3\envs\virtualenv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
c:\Users\krpri\anaconda3\envs\virtualenv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


    epoch  1 | val mean HR=0.1988
127/127 - 22s - 176ms/step - loss: 0.3032 - val_loss: 0.2411 - learning_rate: 0.0010
Epoch 2/40
    epoch  2 | val mean HR=0.2068
127/127 - 17s - 138ms/step - loss: 0.2438 - val_loss: 0.2300 - learning_rate: 0.0010
Epoch 3/40
    epoch  3 | val mean HR=0.2267
127/127 - 17s - 134ms/step - loss: 0.2305 - val_loss: 0.2218 - learning_rate: 0.0010
Epoch 4/40
    epoch  4 | val mean HR=0.2443
127/127 - 17s - 136ms/step - loss: 0.2185 - val_loss: 0.2168 - learning_rate: 0.0010
Epoch 5/40
    epoch  5 | val mean HR=0.2487
127/127 - 17s - 136ms/step - loss: 0.2071 - val_loss: 0.2143 - learning_rate: 0.0010
Epoch 6/40
    epoch  6 | val mean HR=0.2483
127/127 - 17s - 134ms/step - loss: 0.1967 - val_loss: 0.2161 - learning_rate: 0.0010
Epoch 7/40
    epoch  7 | val mean HR=0.2397
127/127 - 17s - 136ms/step - loss: 0.1868 - val_loss: 0.2188 - learning_rate: 0.0010
Epoch 8/40
    epoch  8 | val mean HR=0.2401
127/127 - 17s - 137ms/step - loss: 0.1750 - val_loss: 0.

In [6]:
raw_scores = to_full(net.model.predict([Xte.astype(np.float32),Hte], batch_size=BATCH_SIZE, verbose=0).astype(np.float32))
rbm_scores = np.load(f'{PRED_DIR}/lstm_test_scores.npy').astype(np.float32)
raw_res = evaluate_predictions(raw_scores, yte, K_VALUES)
rbm_res = evaluate_predictions(rbm_scores, yte, K_VALUES)
with open(f'{REPORT_DIR}/phase3_baseline_report.json') as f:
    base = json.load(f)

print(f"{'K':>3} | {'Model':<13} | {'HitRatio':>8} | {'Recall':>8}")
for k in K_VALUES:
    rows = [
        ('Raw history', raw_res[k]),
        ('RBM LSTM', rbm_res[k]),
        ('Most Popular', base['most_popular'][str(k)]),
        ('FP-Growth', base['fpgrowth'][str(k)]),
    ]
    for name, r in rows:
        print(f"{k:>3} | {name:<13} | {r['HitRatio']:>8.4f} | {r['Recall']:>8.4f}")

  K | Model         | HitRatio |   Recall
  1 | Raw history   |   0.1263 |   0.0474
  1 | RBM LSTM      |   0.1119 |   0.0417
  1 | Most Popular  |   0.1533 |   0.0707
  1 | FP-Growth     |   0.1533 |   0.0707
  3 | Raw history   |   0.2473 |   0.1018
  3 | RBM LSTM      |   0.2377 |   0.0975
  3 | Most Popular  |   0.1998 |   0.0927
  3 | FP-Growth     |   0.2034 |   0.0930
  5 | Raw history   |   0.3131 |   0.1334
  5 | RBM LSTM      |   0.3113 |   0.1348
  5 | Most Popular  |   0.2347 |   0.1044
  5 | FP-Growth     |   0.2374 |   0.1051
  9 | Raw history   |   0.3934 |   0.1763
  9 | RBM LSTM      |   0.3843 |   0.1724
  9 | Most Popular  |   0.2796 |   0.1219
  9 | FP-Growth     |   0.2829 |   0.1234


In [7]:
def variance_split(scores):
    s = scores[:, 1:]
    item_mean = s.mean(axis=0)
    total = s.var()
    return item_mean.var() / total, (s - item_mean).var() / total


for name, sc in [('Raw history', raw_scores), ('RBM LSTM', rbm_scores)]:
    item_share, user_share = variance_split(sc)
    print(f"{name}: item identity {item_share:.2%} | user {user_share:.2%}")


def hit_ratio(scores, y, k, mask):
    s = scores[mask].copy()
    s[:, 0] = -np.inf
    idx = np.argsort(-s, axis=1, kind='stable')[:, :k]
    yy = y[mask].copy()
    yy[:, 0] = 0
    hits = np.take_along_axis(yy, idx, axis=1).sum(axis=1) > 0
    valid = yy.sum(axis=1) > 0
    return float(hits[valid].mean())


mp = np.load(f'{PRED_DIR}/most_popular_scores.npy').astype(np.float32)
mp = mp if mp.shape[0] == len(users) else mp[users]
hist_len = (X_padded[users].sum(axis=2) > 0).sum(axis=1)
bins = [(1, 2), (3, 5), (6, 9), (10, 10)]
print(f"{'History':>8} | {'K':>3} | {'Raw':>8} | {'RBM':>8} | {'Most Pop':>8}")
for lo, hi in bins:
    m = (hist_len >= lo) & (hist_len <= hi)
    for k in [3, 9]:
        print(f"{lo}-{hi:>6} | {k:>3} | {hit_ratio(raw_scores, yte, k, m):>8.4f} | "
              f"{hit_ratio(rbm_scores, yte, k, m):>8.4f} | {hit_ratio(mp, yte, k, m):>8.4f}")

Raw history: item identity 41.06% | user 58.94%
RBM LSTM: item identity 66.47% | user 33.53%
 History |   K |      Raw |      RBM | Most Pop
1-     2 |   3 |   0.2063 |   0.2027 |   0.1574
1-     2 |   9 |   0.3576 |   0.3446 |   0.2449
3-     5 |   3 |   0.2440 |   0.2101 |   0.1946
3-     5 |   9 |   0.3954 |   0.3936 |   0.2769
6-     9 |   3 |   0.2924 |   0.3191 |   0.2578
6-     9 |   9 |   0.4449 |   0.4422 |   0.3298
10-    10 |   3 |   0.2779 |   0.2803 |   0.2212
10-    10 |   9 |   0.3882 |   0.3561 |   0.2890


In [8]:
np.save(f'{PRED_DIR}/raw_lstm_test_scores.npy', raw_scores)
net.model.save(f'{MODEL_DIR}/raw_lstm.keras')
with open(f'{REPORT_DIR}/raw_lstm_report.json', 'w') as f:
    json.dump({'config': cfg, 'test_raw_lstm': {str(k): v for k, v in raw_res.items()}}, f, indent=2)
print("Saved raw-history scores, model and report")

Saved raw-history scores, model and report
